# Configuração dos parâmetros

In [0]:
catalog = "olist_medalion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Funções de limpeza e checagem da qualidade dos dados

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window

dq_results = []

# Checagem de qualidade: conta quantas linhas violam a condição esperada.
def dq_check(table_name: str, check_name: str, df, condition):
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

# Checagem de qualidade específica para unicidade de chave.
def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
TEXTOS_AUSENCIA = [
    "", "unknown", "não informado", "nao informado", "n/a", "na", "nan",
    "null", "none", "nulo", "-", "--", "?", "undefined", "sem informação",
    "sem informacao",
]

_TIPOS_INTEIROS = {"int", "integer", "bigint", "long", "smallint", "tinyint"}

def limpar_numero(
    df,
    col_in,
    col_out,
    tipo="decimal(18,2)",
    milhar_ambiguo=False,
    minimo=None,
    maximo=None,
    minimo_exclusivo=False,
):
    
    # Higieniza e converte uma coluna textual para número, sem nunca interromper o pipeline.
    #
    # Parâmetros
    # ----------
    # col_in, col_out    : coluna de origem e de destino (podem ser iguais).
    # tipo               : tipo final. Ex.: "decimal(18,2)"
    # milhar_ambiguo     : True  = "1.234" / "1,234" (1 separador + 3 dígitos) = milhar
    #                     False = "12.345" / "12,345" = decimal
    # minimo / maximo    : limites de negócio (inclusivos). Fora do intervalo = NULL.
    # minimo_exclusivo   : True  = exige valor > minimo.
    # 
    # Regras fixas
    # ------------
    # - Textos de ausência = NULL antes da conversão.
    # - Aceita prefixo de moeda, espaços, sinal, dígitos, "." e ",".
    #  Qualquer outro texto (ex.: column shift) = NULL
    # - Separadores: 1,234.56 | 1.234,56 | 1.234.567 | 1,234,567 | 12,5 | notação científica.
    # - Conversão final com try_cast (overflow também vira NULL).

    raw = F.trim(F.col(col_in).cast("string"))
    raw = F.when(F.lower(raw).isin(TEXTOS_AUSENCIA), F.lit(None)).otherwise(raw)

    df = df.withColumn("_raw", raw)
    df = df.withColumn("_s", F.regexp_replace("_raw", r"(?i)^(us\$|r\$|\$|€|£)|\s+", ""))

    sci = F.col("_s").rlike(r"^-?\d+(\.\d+)?[eE][+-]?\d+$")
    formato_valido = F.col("_s").rlike(r"^-?[0-9.,]+$") | sci

    qtd = "+" if milhar_ambiguo else "{2,}"
    pad_milhar_virg = rf"^-?\d{{1,3}}(,\d{{3}}){qtd}$"
    pad_milhar_pont = rf"^-?\d{{1,3}}(\.\d{{3}}){qtd}$"

    pos_dot = F.instr(F.reverse(F.col("_s")), ".")
    pos_com = F.instr(F.reverse(F.col("_s")), ",")
    tem_dot = pos_dot > 0
    tem_com = pos_com > 0

    parsed = (
        F.when(tem_dot & tem_com & (pos_dot < pos_com),               # 1,234.56
               F.regexp_replace("_s", ",", ""))
         .when(tem_dot & tem_com & (pos_com < pos_dot),               # 1.234,56
               F.regexp_replace(F.regexp_replace("_s", r"\.", ""), ",", "."))
         .when(tem_com & ~tem_dot & F.col("_s").rlike(pad_milhar_virg),
               F.regexp_replace("_s", ",", ""))                       # milhar com vírgula
         .when(tem_com & ~tem_dot,
               F.regexp_replace("_s", ",", "."))                      # vírgula decimal
         .when(tem_dot & ~tem_com & F.col("_s").rlike(pad_milhar_pont),
               F.regexp_replace("_s", r"\.", ""))                     # milhar com ponto
         .otherwise(F.col("_s"))                                      # ponto decimal / inteiro puro
    )

    df = df.withColumn(
        "_clean",
        F.when(~formato_valido, F.lit(None)).when(sci, F.col("_s")).otherwise(parsed),
    )
    df = df.withColumn("_num", F.expr("try_cast(_clean as decimal(30,6))"))

    # regras de negócio
    val = F.col("_num")
    cond = F.lit(True)
    if minimo is not None:
        cond = cond & ((val > minimo) if minimo_exclusivo else (val >= minimo))
    if maximo is not None:
        cond = cond & (val <= maximo)
    if tipo.lower() in _TIPOS_INTEIROS:
        cond = cond & (val == F.floor(val))

    df = df.withColumn(col_out, F.when(cond, F.expr(f"try_cast(_num as {tipo})")))
    return df.drop("_raw", "_s", "_clean", "_num")

## Info_Filmes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Deduplicação: mantém apenas o registro mais recente por filme (id)
w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc_nulls_last())

df_dedup = (
    df_bronze_info
    .filter(F.col("id").isNotNull())
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Status: normalização (ruído, hífens, caixa)
status_norm = F.trim(
    F.regexp_replace(                                   # colapsa espaço múltiplos
        F.regexp_replace(                               # remove tudo que não é letra/espaço
            F.regexp_replace(                           # hífens/underscores -> espaço
                F.lower(F.trim(F.col("status").cast("string"))),
                r"[-_]+", " "),
            r"[^a-z ]", ""),
        r"\s+", " ")
)

# Tradução
status_map = F.create_map(
    F.lit("released"),        F.lit("Lançado"),
    F.lit("post production"), F.lit("Pós-Produção"),
    F.lit("in production"),   F.lit("Em Produção"),
    F.lit("planned"),         F.lit("Planejado"),
    F.lit("rumored"),         F.lit("Rumores"),
    F.lit("canceled"),        F.lit("Cancelado"),
    F.lit("cancelled"),       F.lit("Cancelado"),
)

# Retorna não informado, caso não houver correspondência
status_traduzido = F.coalesce(status_map[status_norm], F.lit("Não Informado"))

# Data multi-formato: NULL só se nenhum funcionar
DATE_FORMATS = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "yyyyMMdd",
    "dd/MM/yyyy",
    "dd-MM-yyyy",
    "dd.MM.yyyy",
    "MM/dd/yyyy",
    "MM-dd-yyyy",
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy-MM-dd'T'HH:mm:ss'Z'",
    "d MMM yyyy",
    "MMM d, yyyy",
    "MMMM d, yyyy",
    "dd MMMM yyyy",
]

date_str = F.trim(F.col("release_date").cast("string"))

def _parse(fmt):
    return F.expr(f"try_to_date(trim(cast(release_date as string)), \"{fmt}\")")

data_lancamento = F.coalesce(*[_parse(f) for f in DATE_FORMATS])

# Silver
df_silver_info = (
    df_dedup
    .withColumn("status", status_traduzido)
    .withColumn("release_date", data_lancamento)
    .withColumn("runtime", F.expr("try_cast(trim(runtime) as int)"))
    .withColumn("title", F.trim(F.col("title")))
    .withColumn("original_title", F.trim(F.col("original_title")))
    .withColumn("original_language", F.lower(F.trim(F.col("original_language"))))
    .withColumn("overview", F.trim(F.col("overview")))
    .withColumn("tagline", F.trim(F.col("tagline")))
    .select("id", "title", "original_title", "release_date", "runtime",
            "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    # Coluna derivada
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

# Checagem de dados
dq_check("silver.movies_info", "id_filme não nulo", df_silver_info, F.col("id_filme").isNotNull())
dq_check_unique("silver.movies_info", "id_filme único", df_silver_info, ["id_filme"])
dq_check("silver.movies_info", "status_filme dentro do domínio", df_silver_info,
         F.col("status_filme").isin("Lançado", "Pós-Produção", "Em Produção",
                                    "Planejado", "Rumores", "Cancelado", "Não Informado"))
dq_check("silver.movies_info", "ano_lancamento coerente com data_lancamento", df_silver_info,
         (F.col("data_lancamento").isNull() & F.col("ano_lancamento").isNull()) |
         (F.col("ano_lancamento") == F.year("data_lancamento")))

df_silver_info \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_silver_info.limit(10))

## Financeiro_Filmes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cotação: última cotação disponível na bronze 
_cotacao = (
    spark.table(f"{catalog}.bronze.tb_cotacao_dolar")
    .withColumn("_dt_cotacao", F.expr("try_to_timestamp(dataHoraCotacao)"))
    .withColumn("_taxa", F.expr("try_cast(cotacaoCompra as double)"))
    .filter(F.col("_dt_cotacao").isNotNull() & (F.col("_taxa") > 0))
    .orderBy(F.col("_dt_cotacao").desc(), F.col("ingestion_datetime").desc_nulls_last())
    .select("_dt_cotacao", "_taxa")
    .limit(1)
    .collect()
)

TAXA_USD_BRL = _cotacao[0]["_taxa"]
print(f"Cotação USD->BRL utilizada: {TAXA_USD_BRL}")

df_bronze_fin = spark.table(f"{bronze_schema}.tb_movies_financials")

# Deduplicação por filme (mantém a ingestão mais recente)
w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc_nulls_last())

df_dedup = (
    df_bronze_fin
    .filter(F.col("id").isNotNull())
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

#  Higienização + conversão
df_clean = limpar_numero(df_dedup, "budget",  "orcamento_usd", tipo="decimal(18,2)",
                   milhar_ambiguo=True, minimo=0, minimo_exclusivo=True)
df_clean = limpar_numero(df_clean, "revenue", "receita_usd",   tipo="decimal(18,2)",
                   milhar_ambiguo=True, minimo=0, minimo_exclusivo=True)

# BRL, lucro e margem
taxa = F.lit(TAXA_USD_BRL).cast("decimal(10,4)")

orc = F.col("orcamento_usd")
rec = F.col("receita_usd")
ambos_nulos = orc.isNull() & rec.isNull()

df_silver_fin = (
    df_clean
    .withColumn("orcamento_brl", F.round(orc * taxa, 2).cast("decimal(18,2)"))
    .withColumn("receita_brl",   F.round(rec * taxa, 2).cast("decimal(18,2)"))
    # Lucro: valor ausente entra como 0 na conta, se ambos ausentes, lucro = NULL
    .withColumn("lucro_usd",
                F.when(ambos_nulos, F.lit(None))
                 .otherwise(F.coalesce(rec, F.lit(0)) - F.coalesce(orc, F.lit(0)))
                 .cast("decimal(18,2)"))
    .withColumn("lucro_brl",
                F.when(ambos_nulos, F.lit(None))
                 .otherwise(F.coalesce(F.col("receita_brl"), F.lit(0))
                            - F.coalesce(F.col("orcamento_brl"), F.lit(0)))
                 .cast("decimal(18,2)"))
    # Margem % = lucro / receita * 100
    .withColumn("margem_lucro_pct",
                F.when(rec > 0, F.round(F.col("lucro_usd") / rec * 100, 2)))
    .withColumnRenamed("id", "id_filme")
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_pct")
)

# Data quality
dq_check("silver.financeiro_filmes", "id_filme não nulo", df_silver_fin, F.col("id_filme").isNotNull())
dq_check_unique("silver.financeiro_filmes", "id_filme único", df_silver_fin, ["id_filme"])
dq_check("silver.financeiro_filmes", "orcamento_usd nulo ou positivo", df_silver_fin,
         F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0))
dq_check("silver.financeiro_filmes", "receita_usd nula ou positiva", df_silver_fin,
         F.col("receita_usd").isNull() | (F.col("receita_usd") > 0))
dq_check("silver.financeiro_filmes", "BRL nulo quando USD nulo", df_silver_fin,
         (F.col("orcamento_usd").isNotNull() | F.col("orcamento_brl").isNull()) &
         (F.col("receita_usd").isNotNull() | F.col("receita_brl").isNull()))
dq_check("silver.financeiro_filmes", "margem nula quando sem receita", df_silver_fin,
         F.col("receita_usd").isNotNull() | F.col("margem_lucro_pct").isNull())

df_silver_fin.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

display(df_silver_fin.limit(10))

## Metrica Engajamento

In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

# Deduplicação por filme 
w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc_nulls_last())

df_dedup = (
    df_bronze_metrics
    .filter(F.col("id").isNotNull())
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Conversão segura
# Popularidade
df_num = limpar_numero(df_dedup, "popularity", "popularidade", tipo="decimal(18,4)", minimo=0)

# Notas de 0 a 10
df_num = limpar_numero(df_num, "vote_average",  "nota_media_tmdb", tipo="decimal(6,3)", minimo=0, maximo=10)
df_num = limpar_numero(df_num, "averageRating", "nota_media_imdb", tipo="decimal(6,3)", minimo=0, maximo=10)

# Contagens de votos
df_num = limpar_numero(df_num, "vote_count", "qtd_votos_tmdb", tipo="bigint",
                   milhar_ambiguo=True, minimo=0)
df_num = limpar_numero(df_num, "numVotes",   "qtd_votos_imdb", tipo="bigint",
                   milhar_ambiguo=True, minimo=0)

# Regras de negócio
#    - notas: somente entre 0 e 10
#    - contagens: inteiros >= 0
#    - popularidade: >= 0
def nota_valida(c):
    return F.when(F.col(c).between(0, 10), F.col(c).cast("decimal(6,3)"))

def contagem_valida(c):
    return F.when((F.col(c) >= 0) & (F.col(c) == F.floor(F.col(c))), F.col(c).cast("bigint"))

df_silver_metrics = (
    df_num
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade").cast("decimal(18,4)")))
    .withColumn("nota_media_tmdb", nota_valida("nota_media_tmdb"))
    .withColumn("nota_media_imdb", nota_valida("nota_media_imdb"))
    .withColumn("qtd_votos_tmdb", contagem_valida("qtd_votos_tmdb"))
    .withColumn("qtd_votos_imdb", contagem_valida("qtd_votos_imdb"))
    .withColumnRenamed("id", "id_filme")
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
            "nota_media_imdb", "qtd_votos_imdb")
)

# Data quality
t = "silver.metricas_engajamento"
dq_check(t, "id_filme não nulo", df_silver_metrics, F.col("id_filme").isNotNull())
dq_check_unique(t, "id_filme único", df_silver_metrics, ["id_filme"])
dq_check(t, "nota_media_tmdb entre 0 e 10", df_silver_metrics,
         F.col("nota_media_tmdb").isNull() | F.col("nota_media_tmdb").between(0, 10))
dq_check(t, "nota_media_imdb entre 0 e 10", df_silver_metrics,
         F.col("nota_media_imdb").isNull() | F.col("nota_media_imdb").between(0, 10))
dq_check(t, "qtd_votos_tmdb >= 0", df_silver_metrics,
         F.col("qtd_votos_tmdb").isNull() | (F.col("qtd_votos_tmdb") >= 0))
dq_check(t, "qtd_votos_imdb >= 0", df_silver_metrics,
         F.col("qtd_votos_imdb").isNull() | (F.col("qtd_votos_imdb") >= 0))
dq_check(t, "popularidade >= 0", df_silver_metrics,
         F.col("popularidade").isNull() | (F.col("popularidade") >= 0))

df_silver_metrics.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_metrics.limit(10))

## Avaliações Usuários

In [0]:
COMENTARIO_PADRAO = "Sem comentário"

df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

# Limpeza e regras de negócio
df_clean = limpar_numero(df_bronze_reviews, "nota", "nota", tipo="decimal(4,2)", minimo=0, maximo=10)

comentario_vazio = (
    F.col("comentario").isNull() |
    (F.length(F.regexp_replace(F.col("comentario").cast("string"), r"[\s\u00A0]+", "")) == 0)
)

df_prep = (
    df_clean
    .withColumn("nome", F.trim(F.col("nome")))
    # nota fora de 0-10 (inclui erro de escala) = NULL
    .withColumn("nota", F.when(F.col("nota").between(0, 10),
                               F.col("nota").cast("decimal(4,2)")))
    # comentário nulo ou só com espaços = texto padrão
    .withColumn("comentario",
                F.when(comentario_vazio, F.lit(COMENTARIO_PADRAO))
                 .otherwise(F.trim(F.col("comentario"))))
)

# Deduplicação: filme + usuário + nota + comentário idênticos
w_dedup = (Window.partitionBy("id", "nome", "nota", "comentario")
           .orderBy(F.col("ingestion_datetime").desc_nulls_last()))

df_silver_reviews = (
    df_prep
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .select("id", "nome", "nota", "comentario")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

# Data quality
t = "silver.avaliacoes_usuarios"
dq_check(t, "id_filme não nulo", df_silver_reviews, F.col("id_filme").isNotNull())
dq_check_unique(t, "avaliação única (filme, usuário, nota, comentário)", df_silver_reviews,
                ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
dq_check(t, "nota_usuario entre 0 e 10", df_silver_reviews,
         F.col("nota_usuario").isNull() | F.col("nota_usuario").between(0, 10))
dq_check(t, "comentario_usuario preenchido", df_silver_reviews,
         F.col("comentario_usuario").isNotNull() &
         (F.length(F.trim(F.col("comentario_usuario"))) > 0))

df_silver_reviews.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_reviews.limit(10))

## Generos

In [0]:
# Domínio de gêneros válidos (lista fechada, conforme a base)
GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

# chave em minúsculas = nome canônico
mapa_generos = F.create_map(*[x for g in GENEROS_VALIDOS for x in (F.lit(g.lower()), F.lit(g))])

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Um registro por filme
w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc_nulls_last())

df_dedup = (
    df_bronze_credits
    .filter(F.col("id").isNotNull())
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .select("id", "genres")
)

# Padroniza separadores ("," / ";" / "|") ANTES do split e explode
genres_txt = F.regexp_replace(F.col("genres").cast("string"), r"[\[\]\"]", "")
genres_txt = F.regexp_replace(genres_txt, r"\s*[;|]\s*", ",")

df_exploded = (
    df_dedup
    .withColumn("genero_raw", F.explode(F.split(genres_txt, r"\s*,\s*")))
    .withColumn("genero_raw", F.regexp_replace("genero_raw", r"^['\s]+|['\s]+$", ""))
    .withColumn("genero_raw", F.regexp_replace("genero_raw", r"\s+", " "))
    .withColumn("genero", mapa_generos[F.lower(F.col("genero_raw"))])
)

# Silver: só gêneros do domínio, únicos por filme
df_silver_generos = (
    df_exploded
    .filter(F.col("genero").isNotNull())
    .select(F.col("id").alias("id_filme"), "genero")
    .dropDuplicates(["id_filme", "genero"])
)

# Data quality
t = "silver.generos"
dq_check(t, "id_filme não nulo", df_silver_generos, F.col("id_filme").isNotNull())
dq_check(t, "genero dentro do domínio", df_silver_generos, F.col("genero").isin(GENEROS_VALIDOS))
dq_check_unique(t, "(id_filme, genero) único", df_silver_generos, ["id_filme", "genero"])

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_generos")

display(df_silver_generos.limit(10))

## Pessoas_empresas

In [0]:
from functools import reduce

# Mapeamento coluna de origem = tipo de entidade
MAPA_TIPOS = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

# Limites para identificar texto descritivo 
MAX_CARACTERES_NOME = 60
MAX_PALAVRAS_NOME = 8

df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Um registro por filme 
w_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc_nulls_last())

df_dedup = (
    df_bronze_credits
    .filter(F.col("id").isNotNull())
    .withColumn("_rn", F.row_number().over(w_dedup))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Função única: split + explode + limpeza de uma coluna de entidades
def explodir_entidades(df, coluna, tipo):
    # padroniza separadores ("," / ";" / "|") antes do split
    txt = F.regexp_replace(F.col(coluna).cast("string"), r"[\[\]\"]", "")
    txt = F.regexp_replace(txt, r"\s*[;|]\s*", ",")

    return (
        df.select("id", F.explode(F.split(txt, r"\s*,\s*")).alias("nome_raw"))
        .withColumn("nome_raw", F.regexp_replace("nome_raw", r"^['\s]+|['\s]+$", ""))  # aspas/espaços nas pontas
        .withColumn("nome_raw", F.regexp_replace("nome_raw", r"\s+", " "))             # espaços repetidos
        .withColumn("tipo_entidade", F.lit(tipo))
    )

df_exploded = reduce(
    lambda a, b: a.unionByName(b),
    [explodir_entidades(df_dedup, col, tipo) for col, tipo in MAPA_TIPOS.items()],
)

# Remove resíduos
#    Nome válido: começa com letra/número, contém ao menos uma letra, usa apenas
#    letras, números, espaço e - & ' . ( ) / ! +, com tamanho/palavras limitados.
n_palavras = F.size(F.split(F.col("nome_raw"), " "))

valido = (
    F.col("nome_raw").isNotNull()
    & ~F.lower(F.col("nome_raw")).isin(TEXTOS_AUSENCIA)
    & F.col("nome_raw").rlike(r"^[\p{L}\p{N}][\p{L}\p{N} \-&'.()/!+]*$")
    & F.col("nome_raw").rlike(r"\p{L}")
    & (F.length("nome_raw") <= MAX_CARACTERES_NOME)
    & (n_palavras <= MAX_PALAVRAS_NOME)
)

df_marcado = df_exploded.withColumn("_valido", valido)

# Capitalização: padroniza apenas textos em maiúsculas ou em minúsculas
nome_padrao = (
    F.when((F.col("nome_raw") == F.upper("nome_raw")) | (F.col("nome_raw") == F.lower("nome_raw")),
           F.initcap("nome_raw"))
     .otherwise(F.col("nome_raw"))
)

# Silver: modelo unificado, sem duplicados
df_silver_entidades = (
    df_marcado
    .filter(F.col("_valido"))
    .withColumn("nome_entidade", nome_padrao)
    .select(F.col("id").alias("id_filme"), "nome_entidade", "tipo_entidade")
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

# Auditoria: o que foi descartado
df_descartados = (
    df_marcado
    .filter(~F.col("_valido") & F.col("nome_raw").isNotNull() & (F.col("nome_raw") != ""))
    .groupBy("tipo_entidade", "nome_raw").count()
    .orderBy(F.desc("count"))
)

# Data quality
t = "silver.pessoas_empresas"
dq_check(t, "id_filme não nulo", df_silver_entidades, F.col("id_filme").isNotNull())
dq_check(t, "nome_entidade preenchido", df_silver_entidades,
         F.col("nome_entidade").isNotNull() & (F.length(F.trim("nome_entidade")) > 0))
dq_check(t, "tipo_entidade dentro do domínio", df_silver_entidades,
         F.col("tipo_entidade").isin(list(MAPA_TIPOS.values())))
dq_check(t, "nome_entidade com ao menos uma letra", df_silver_entidades,
         F.col("nome_entidade").rlike(r"\p{L}"))
dq_check_unique(t, "(id_filme, nome_entidade, tipo_entidade) único", df_silver_entidades,
                ["id_filme", "nome_entidade", "tipo_entidade"])

df_silver_entidades.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(df_silver_entidades.limit(10))

## Cotação Dolar

In [0]:
df_bronze_raw = spark.table(f"{catalog}.bronze.tb_cotacao_dolar")

df_cleaned = limpar_numero(df_bronze_raw, "cotacaoCompra", "valor_cotacao", "decimal(18,4)",
    minimo=0, minimo_exclusivo=True
)

# Obtenção do último registro diário disponível
df_daily = (
    df_cleaned
    .withColumn("ts_cotacao", F.expr("try_to_timestamp(dataHoraCotacao)"))
    .withColumn("dt_cotacao", F.to_date("ts_cotacao"))
    .filter(F.col("dt_cotacao").isNotNull() & F.col("valor_cotacao").isNotNull())
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("dt_cotacao")
                  .orderBy(F.col("ts_cotacao").desc(), F.col("ingestion_datetime").desc_nulls_last())
        ),
    )
    .filter(F.col("rn") == 1)
    .select("dt_cotacao", "valor_cotacao")
)

# Construção da Série Temporal Contínua
limites = df_daily.select(
    F.min("dt_cotacao").alias("min_dt"),
    F.max("dt_cotacao").alias("max_dt")
).collect()[0]

df_calendar = spark.range(1).select(
    F.explode(
        F.sequence(
            F.lit(limites["min_dt"]).cast("date"),
            F.lit(limites["max_dt"]).cast("date"),
            F.expr("interval 1 day")
        )
    ).alias("dt")
)

# Join + Forward Fill
w_ffill = Window.orderBy("dt").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver = (
    df_calendar
    .join(df_daily, df_calendar.dt == df_daily.dt_cotacao, "left")
    .withColumn("cotacao_dolar", F.last("valor_cotacao", ignorenulls=True).over(w_ffill))
    .withColumn("eh_dia_util", F.col("valor_cotacao").isNotNull())
    .select(
        F.col("dt").alias("data_cotacao"),
        F.col("cotacao_dolar"),
        F.col("eh_dia_util")
    )
)

# Data quality
t = "silver.tb_cotacao_dolar"

dq_check(t, "data_cotacao não nula", df_silver, F.col("data_cotacao").isNotNull())
dq_check(t, "cotacao_dolar não nula", df_silver, F.col("cotacao_dolar").isNotNull())
dq_check(t, "cotacao_dolar maior que zero", df_silver, F.col("cotacao_dolar") > 0)
dq_check(t, "eh_dia_util preenchido", df_silver, F.col("eh_dia_util").isNotNull())
dq_check_unique(t, "data_cotacao única", df_silver, ["data_cotacao"])

df_silver.write.format("delta").mode("append").saveAsTable(f"{catalog}.silver.tb_cotacao_dolar")

display(df_silver.limit(10))